In [1]:
import keras
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

In [2]:
DATA_DIR = "mu3e_trigger_data"
SIGNAL_DATA_FILE = f"{DATA_DIR}/run42_sig_positions.npy"
BACKGROUND_DATA_FILE = f"{DATA_DIR}/run42_bg_positions.npy"

max_barrel_radius = 86.3
max_endcap_distance = 372.6

In [3]:
signal_data = np.load(SIGNAL_DATA_FILE)
background_data = np.load(BACKGROUND_DATA_FILE)

background_data[background_data[:, :, 0] != -1, 0] = (
    background_data[background_data[:, :, 0] != -1, 0]
) / max_barrel_radius
background_data[background_data[:, :, 0] != -1, 1] = (
    background_data[background_data[:, :, 0] != -1, 1]
) / max_barrel_radius
background_data[background_data[:, :, 0] != -1, 2] = (
    background_data[background_data[:, :, 0] != -1, 2]
) / max_endcap_distance

signal_data[signal_data[:, :, 0] != -1, 0] = (
    signal_data[signal_data[:, :, 0] != -1, 0]
) / max_barrel_radius
signal_data[signal_data[:, :, 0] != -1, 1] = (
    signal_data[signal_data[:, :, 0] != -1, 1]
) / max_barrel_radius
signal_data[signal_data[:, :, 0] != -1, 2] = (
    signal_data[signal_data[:, :, 0] != -1, 2]
) / max_endcap_distance

In [4]:
bg_mask = background_data[:, :, 0] != -1
signal_mask = signal_data[:, :, 0] != -1
bg_seq_length = bg_mask.sum(axis=1)
signal_seq_length = signal_mask.sum(axis=1)

background_data = background_data[bg_seq_length > 0]
signal_data = signal_data[signal_seq_length > 0]

In [5]:
sequence_length = signal_data.shape[1]
input_feature_dim = signal_data.shape[2]
feature_dim = 16

In [6]:
def cartesian_to_cylindrical(data):
    r = np.sqrt(data[:, :, 0]**2 + data[:, :, 1]**2)
    theta = np.arctan2(data[:, :, 1], data[:, :, 0])
    z = data[:, :, 2]
    cylindrical_data = np.stack((r, theta, z), axis=-1)
    cylindrical_data[data[:, :, 0] == -1] = -1
    return cylindrical_data

signal_data_cylindrical = cartesian_to_cylindrical(signal_data)
background_data_cylindrical = cartesian_to_cylindrical(background_data)

In [7]:
from src.model.components import SelfAttentionStack, MLP, GenerateMask, PointTransformer

encoder_input = keras.Input(
    shape=(sequence_length, input_feature_dim), name="encoder_input"
)
mask = GenerateMask(padding_value=-1)(encoder_input)


input_embedding = MLP(
    feature_dim, num_layers=3, hidden_activation="relu", activation="linear", name="input_embedding_mlp"
)(encoder_input)

attention_output = SelfAttentionStack(
    num_heads=4,
    key_dim=feature_dim,
    dropout_rate=0.1,
    stack_size=3,
    name="self_attention_stack",
)(input_embedding, mask)

output_embedding = MLP(
    feature_dim, num_layers=3, hidden_activation="relu", activation="linear", name="output_embedding"
)(attention_output)

encoded_output = keras.layers.GlobalAveragePooling1D(name="global_average_pooling")(
    output_embedding, mask=mask
)

encoder = keras.Model(
    inputs=encoder_input,
    outputs=encoded_output,
    name="encoder",
)

In [ ]:
from src.model import AutoEncoder
autoencoder = AutoEncoder(
    input_size = feature_dim,
    latent_size =  feature_dim //2,
    num_layers = 3
)(encoder_input)

In [ ]:
from src.training import MultiObjectiveTrainer
trainer = MultiObjectiveTrainer(
    encoder=encoder,
    autoencoder=autoencoder,
    lambda_var=1.0,
)
encoder_optimizer = keras.optimizers.Adam(learning_rate=0.001)
ae_optimizer = keras.optimizers.Adam(learning_rate=0.01)

In [10]:
from sklearn.model_selection import train_test_split

bg_train, bg_test = train_test_split(background_data_cylindrical[:10000], test_size=0.2, random_state=42)

bg_train = tf.data.Dataset.from_tensor_slices(bg_train).batch(512, drop_remainder=True)
latent_dataset = bg_train.map(encoder, num_parallel_calls=tf.data.AUTOTUNE)


In [ ]:
for var_epoch in range(10):
    print(f"  Variance Epoch {var_epoch + 1}")
    losses = trainer.train_encoder_variance_step(bg_train, encoder_optimizer)
    print(f"    Losses: {losses}")

for epoch in range(5):
    print(f"Epoch {epoch + 1}")
    print(f"  Autoencoder Epochs")
    losses = trainer.train_autoencoder_step(bg_train, ae_optimizer, num_steps=10)
    for loss in losses:
        print(f"    Loss: {loss}")

    for recon_epoch in range(4):
        print(f"  Reconstruction Epoch {recon_epoch + 1}")
        losses = trainer.train_encoder_step(bg_train, encoder_optimizer)
        print(f"    Losses: {losses}")

  Variance Epoch 1
    Losses: 0.8223934769630432
  Variance Epoch 2
    Losses: 0.660987138748169
  Variance Epoch 3
    Losses: 0.6055777072906494
  Variance Epoch 4
    Losses: 0.5709958672523499
  Variance Epoch 5
    Losses: 0.49872711300849915
  Variance Epoch 6
    Losses: 0.4628259837627411
  Variance Epoch 7
    Losses: 0.42968374490737915
  Variance Epoch 8
    Losses: 0.3921334147453308
  Variance Epoch 9
    Losses: 0.3798759877681732
  Variance Epoch 10
    Losses: 0.3741510212421417
Epoch 1
  Autoencoder Epochs
    Loss: 1.6961110830307007
    Loss: 0.5314550995826721
    Loss: 0.4307062029838562
    Loss: 0.33310458064079285
    Loss: 0.23684808611869812
    Loss: 0.14836671948432922
    Loss: 0.08933742344379425
    Loss: 0.056109409779310226
    Loss: 0.040827490389347076
    Loss: 0.034499093890190125
  Reconstruction Epoch 1
    Losses: (<tf.Tensor: shape=(), dtype=float32, numpy=0.024609075859189034>, <tf.Tensor: shape=(), dtype=float32, numpy=0.3719821572303772>)
 

In [ ]:
trainer.train_autoencoder_step(bg_train, ae_optimizer, num_steps=10)

In [ ]:
bg_test_embeddings = encoder.predict(bg_test)
signal_embeddings = encoder.predict(signal_data_cylindrical[:2000])
bg_train_embeddings = encoder.predict(bg_train)

In [ ]:
bg_test_diff = bg_test_embeddings - autoencoder.predict(bg_test_embeddings)
signal_diff = signal_embeddings - autoencoder.predict(signal_embeddings)
bg_train_diff = bg_train_embeddings - autoencoder.predict(bg_train_embeddings)

In [ ]:
fig, ax_array = plt.subplots(figsize=(10, 5), nrows=4, ncols=4)
for i in range(16):
    ax = ax_array[i // 4, i % 4]
    bg_ad_score = bg_test_diff[:, i]
    signal_ad_score = signal_diff[:, i]
    bins = np.linspace(
        min(np.min(bg_ad_score), np.min(signal_ad_score)),
        max(np.max(bg_ad_score), np.max(signal_ad_score)),
        30,
    )
    ax.hist(
        bg_ad_score,
        bins=bins,
        alpha=0.5,
        label="Background Data",
        color="blue",
        density=True,
    )
    ax.hist(
        signal_ad_score,
        bins=bins,
        alpha=0.5,
        label="Signal Data",
        color="red",
        density=True,
    )
    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_xlabel(f" $\sigma_{i+1}$")
    ax.set_ylabel("Density")
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2)
fig.savefig("ad_scores_train_test.png", dpi=300, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bg_ad_score = np.linalg.norm(bg_test_diff, axis=1)
signal_ad_score = np.linalg.norm(signal_diff, axis=1)
bins = np.linspace(
    min(np.min(bg_ad_score), np.min(signal_ad_score)),
    max(np.max(bg_ad_score), np.max(signal_ad_score)),
    30,
)
ax.hist(
    bg_ad_score,
    bins=bins,
    alpha=0.5,
    label="Background Data",
    color="blue",
    density=True,
)
ax.hist(
    signal_ad_score,
    bins=bins,
    alpha=0.5,
    label="Signal Data",
    color="red",
    density=True,
)

ax.set_xlabel(f" $\sigma_{i+1}$")
ax.set_ylabel("Density")
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2)
fig.savefig("ad_scores_train_test.png", dpi=300, bbox_inches="tight")

In [ ]:
mu = np.mean(bg_train_diff, axis=0)
sigma = np.std(bg_train_diff, axis=0)

dummy_data = np.random.normal(loc=mu, scale=sigma, size=(1000, feature_dim))
dummy_data_diff = dummy_data - autoencoder.predict(dummy_data)